Workspaces > How to Work with All of Us Genomic Data (Hail - Plink)(v8) > Analysis > 01_Get Started with WGS Data_part1_srWGS SNPINDEL

In [1]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables.ipynb

Found bucket: id=rw-migration-aou-rw-f7a4d148, bucketName=rw-migration-aou-rw-f7a4d148
-> Assigned migration variables (ID: rw-migration-aou-rw-f7a4d148)
Found bucket: id=temporary-workspace-bucket, bucketName=temporary-workspace-bucket-wb-perky-cabbage-8342
Found bucket: id=workspace-bucket, bucketName=workspace-bucket-wb-perky-cabbage-8342
✅ Successfully identified latest dataset: wb-silky-artichoke-2408.C2024Q3R9

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-perky-cabbage-8342
WORKSPACE_BUCKET: gs://workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_TEMP_BUCKET: gs://temporary-workspace-bucket-wb-perky-cabbage-8342
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9
bucket_aou_tutorial: NOT FOUND
bucket_id_aou_tutorial: NOT FOUND
bucket_migrated: gs://rw-migration-aou-rw-f7a4d148
bucket_id_migrated: rw-migration-aou-rw-f7a4d148

✅ Saved to /home/jupyter/.bashrc
C2024Q3R9 BQ_DATASET
Multi-trait-GWAS-in-admixed-populations GIT_REPO
dataset_test2 BQ_DATASET
prep_C2024Q3R9 BQ_DATASET
rw-mig

In [2]:
%run /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Setting_Env_Variables_p2.ipynb

WORKSPACE_CDR = wb-silky-artichoke-2408.C2024Q3R9
WORKSPACE_BUCKET = gs://workspace-bucket-wb-perky-cabbage-8342
GOOGLE_PROJECT = wb-perky-cabbage-8342
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Tests/Setting_Env_Variables_p2.R
Done! 10 variables saved to: /home/jupyter/repos/Multi-trait-GWAS-in-admixed-populations/notebooks/Tests/Setting_Env_Variables.sas


# Importation des données `ancestry_pred`

In [3]:
import os
import subprocess
import pandas as pd

In [58]:
!gcloud storage cp gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv . --billing-project=$$GOOGLE_PROJECT

Copying gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv to file://./echo_v4_r2.ancestry_preds.tsv
  Completed files 1/1 | 163.4MiB/163.4MiB                                      

Average throughput: 483.3MiB/s


## Accès aux données controlées du `CDR`

Lien vers CDR pour l'accès aux données génétiques -> https://support.researchallofus.org/hc/en-us/articles/360033200232-Data-Dictionaries

In [62]:
# Définit le chemin vers les fichiers auxiliaires SNP/Indel dans le bucket Google Cloud
auxiliary_path = "gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux"
auxiliary_path

'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux'

In [63]:
# Définit le chemin vers le dossier contenant les données d'ascendance
ancestry_path = f'{auxiliary_path}/ancestry'
ancestry_path

'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry'

In [64]:
# Définit le chemin vers le fichier TSV contenant les prédictions d'ascendance
ancestry_pred_path = f'{ancestry_path}/echo_v4_r2.ancestry_preds.tsv'
ancestry_pred_path

'gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv'

## Extraction des données via `hail`

`Hail` est une librairie utilisée pour l'analyse génétique à grande échelle.

Nous définissons ici `GRCh38` comme la référence génomique par défaut. Cela permet de s'assurer que toutes les opérations utilisant une référence génomique (par exemple, l'importation de variants ou l'annotation génomique) utiliseront le génome `GRCh38`.

In [46]:
!gsutil -u $$GOOGLE_PROJECT cat gs://vwb-aou-datasets-controlled/v9/microarray/vcf/manifest.csv | head -n 10

person_id,vcf_uri,vcf_index_uri
1000000,gs://vwb-aou-datasets-controlled/pooled/microarray/vcf/v8_base/1000000.21317004391.sorted.vcf.gz,gs://vwb-aou-datasets-controlled/pooled/microarray/vcf/v8_base/1000000.21317004391.sorted.vcf.gz.tbi
1000004,gs://vwb-aou-datasets-controlled/pooled/microarray/vcf/v8_base/1000004.20364003917.sorted.vcf.gz,gs://vwb-aou-datasets-controlled/pooled/microarray/vcf/v8_base/1000004.20364003917.sorted.vcf.gz.tbi
1000033,gs://vwb-aou-datasets-controlled/pooled/microarray/vcf/v8_base/1000033.20332000307.sorted.vcf.gz,gs://vwb-aou-datasets-controlled/pooled/microarray/vcf/v8_base/1000033.20332000307.sorted.vcf.gz.tbi
1000036,gs://vwb-aou-datasets-controlled/pooled/microarray/vcf/v9_delta/1000036.22244002782.sorted.vcf.gz,gs://vwb-aou-datasets-controlled/pooled/microarray/vcf/v9_delta/1000036.22244002782.sorted.vcf.gz.tbi
1000039,gs://vwb-aou-datasets-controlled/pooled/microarray/vcf/v8_base/1000039.21050000759.sorted.vcf.gz,gs://vwb-aou-datasets-controlled/pool

In [54]:
!gsutil -m -u $$GOOGLE_PROJECT ls gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv
gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.preds_oth.html
gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/eigenvalues.txt
gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/merged_sites_only_intersection.vcf.bgz
gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/merged_sites_only_intersection.vcf.bgz.tbi
gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/rf_classifier.pkl
gs://vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/training_pca.tsv
gs://vwb-aou-datasets-controlled/v8/wgs/s

In [60]:
!wb resource list

ID                                       RESOURCE TYPE  STEWARDSHIP TYPE  DESCRIPTION                             
C2024Q3R9                                BQ_DATASET     REFERENCED        (unset)                                 
Multi-trait-GWAS-in-admixed-populations  GIT_REPO       REFERENCED        Paper Project                           
dataset_test2                            BQ_DATASET     CONTROLLED        bigquery dataset test COPY_REFERENCE    
prep_C2024Q3R9                           BQ_DATASET     REFERENCED        (unset)                                 
rw-migration-aou-rw-f7a4d148             GCS_BUCKET     CONTROLLED        RW migration bucket for workspace aou...
temporary-workspace-bucket               GCS_BUCKET     CONTROLLED        Bucket for temporary storage of file ...
workspace-bucket                         GCS_BUCKET     CONTROLLED        Primary workspace bucket for storing ...


In [70]:
pd.read_csv('echo_v4_r2.ancestry_preds.tsv', sep='\t')

,research_id,ancestry_pred,probabilities,pca_features,ancestry_pred_other
0,1000000,afr,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.29356093859992993, -0.006344531427051658, ...",afr
1,1000004,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]","[0.10130837407349723, 0.13870298220174238, 0.0...",eur
2,1000033,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]","[0.09848604976046149, 0.1245991833533566, 0.00...",eur
3,1000039,afr,"[1.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[-0.26713841551900935, 0.00469795004195689, 0....",afr
4,1000042,afr,"[0.99, 0.01, 0.0, 0.0, 0.0, 0.0]","[-0.2562773941608334, 0.004901392894403225, -0...",afr
...,...,...,...,...,...
414825,9999678,eur,"[0.0, 0.0, 0.0, 0.99, 0.0, 0.01]","[0.09874274635571259, 0.13176851496404074, 0.0...",eur
414826,9999697,amr,"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0]","[0.08506698802084682, 0.028602321968432515, -0...",amr
414827,9999715,eur,"[0.0, 0.0, 0.0, 1.0, 0.0, 0.0]","[0.0995298469352078, 0.13283576852436813, 0.00...",eur
414828,9999755,eur,"[0.02, 0.17, 0.02, 0.64, 0.12, 0.03]","[0.06449853461869955, 0.11663708597332126, 0.0...",oth


In [65]:
import hail as hl

# Définit GRCh38 comme référence génomique par défaut pour Hail
hl.default_reference(new_default_reference = "GRCh38")

# Importe un fichier contenant les prédictions d'ascendance, en inférant les types sauf pour les colonnes spécifiées
ancestry_pred = hl.import_table(ancestry_pred_path,
                               key="research_id",
                               impute=True,
                               types={"research_id":"tstr","pca_features":hl.tarray(hl.tfloat)})

FatalError: UnsupportedFileSystemException: No FileSystem for scheme "gs"

Java stack trace:
org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "gs"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3443)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at is.hail.io.fs.HadoopFSURL.<init>(HadoopFS.scala:77)
	at is.hail.io.fs.HadoopFS.parseUrl(HadoopFS.scala:91)
	at is.hail.io.fs.HadoopFS.parseUrl(HadoopFS.scala:87)
	at is.hail.io.fs.FS.glob(FS.scala:333)
	at is.hail.io.fs.FS.glob$(FS.scala:333)
	at is.hail.io.fs.HadoopFS.glob(HadoopFS.scala:87)
	at is.hail.io.fs.HadoopFS.$anonfun$globAl

## Load 16 PCA and save to bucket

In [ ]:
# Calcule le nombre maximal de composantes principales (PCA) présentes dans les prédictions d'ascendance
n_pcs = ancestry_pred.aggregate(hl.agg.max(hl.len(ancestry_pred.pca_features)))

In [ ]:
# Pour chaque composante principale, crée une nouvelle colonne PC1, PC2, ..., avec la valeur correspondante extraite de pca_features
for i in range(n_pcs):
    ancestry_pred = ancestry_pred.annotate(**{f"PC{i+1}": ancestry_pred.pca_features[i]})

In [ ]:
# Enlève la clé actuelle sur la table ancestry_pred pour créer une table sans clé
ancestry_pred_unkeyed = ancestry_pred.key_by()

# Prépare la liste des colonnes à exporter : 'research_id' plus toutes les colonnes PC1, PC2, ..., PCn
pc_columns = ['research_id'] + [f"PC{i+1}" for i in range(n_pcs)]

# Sélectionne ces colonnes dans la table sans clé et convertit le résultat en DataFrame pandas
df_pca = ancestry_pred_unkeyed.select(*pc_columns).to_pandas()
df_pca

In [ ]:
df_pca.to_csv("ancestry_pcs.tsv", sep="\t", index=False)

# Importation des données extraites 

In [ ]:
import os
import subprocess

# Get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
print(subprocess.check_output(f"gsutil ls -r {my_bucket}", shell=True).decode('utf-8'))

In [8]:
name_of_file_in_bucket = 'ancestry_pcs.tsv'

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
my_dataframe = pd.read_csv(name_of_file_in_bucket, sep='\t')
my_dataframe

Copying gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/ancestry_pcs.tsv...
\ [1 files][140.8 MiB/140.8 MiB]                                                
Operation completed over 1 objects/140.8 MiB.                                    


[INFO] ancestry_pcs.tsv is successfully downloaded into your working space


,research_id,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,PC16
0,1000000,-0.293561,-0.006345,0.002386,0.001446,0.024304,-0.001529,-0.005621,-0.001204,-0.000920,0.006886,0.004709,0.004915,0.011461,0.002675,-0.001492,0.000896
1,1000004,0.101308,0.138703,0.006683,0.053012,0.003345,0.019714,-0.011616,-0.001016,-0.001095,-0.000909,-0.001277,-0.000694,-0.000668,-0.000960,-0.001257,0.000115
2,1000033,0.098486,0.124599,0.009398,0.042617,0.003846,0.026659,-0.018481,-0.001463,-0.001203,0.000465,0.000469,0.000629,0.000019,-0.000210,-0.000013,0.000411
3,1000039,-0.267138,0.004698,0.001046,0.001767,0.031375,0.001713,-0.009450,0.016385,-0.000844,0.001511,0.003544,-0.000247,-0.010346,0.005279,-0.013716,0.008376
4,1000042,-0.256277,0.004901,-0.002448,0.009514,0.008931,0.010683,0.003820,-0.002210,0.010388,-0.008314,-0.002740,0.003094,0.007370,0.001563,-0.002620,0.006765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
414825,9999678,0.098743,0.131769,0.010411,0.048562,0.002988,0.021923,-0.017846,-0.002502,-0.000634,0.000194,0.000252,-0.001290,0.001715,-0.000752,-0.000415,-0.000020
414826,9999697,0.085067,0.028602,-0.107409,0.005666,0.000069,-0.012026,0.010985,0.002458,0.001513,-0.001535,-0.001271,-0.001320,-0.000950,-0.001408,-0.003327,0.000309
414827,9999715,0.099530,0.132836,0.008332,0.050046,0.003166,0.023643,-0.015716,-0.002708,-0.001984,-0.001566,0.000294,0.000889,-0.001221,0.001146,-0.000401,0.000165
414828,9999755,0.064499,0.116637,0.009492,0.042075,-0.005982,-0.022958,0.020151,0.001276,0.000953,-0.000316,-0.003218,0.000905,-0.000271,0.000847,0.001971,-0.000443


# Get admixture with Rye

In [27]:
admixture_estimates_q = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/admixture_estimates/aou_admixture_estimates_rye_v8.Q"
!gsutil -m -u $$GOOGLE_PROJECT cat $admixture_estimates_q | head -n 3

AccessDeniedException: 403 pet-2766430754533474dc47a@wb-perky-cabbage-8342.iam.gserviceaccount.com does not have storage.objects.list access to the Google Cloud Storage bucket. Permission 'storage.objects.list' denied on resource '//storage.googleapis.com/projects/_/buckets/fc-aou-datasets-controlled' (or it may not exist). Remediate access with this Troubleshooter URL or share it with your administrator - https://console.cloud.google.com/iam-admin/troubleshooter/summary;errorId=CiQwMTlmMzZjNi0yMjMyLTdmYTUtOTE4YS02ODk3YzE0YThiNmQSLXByb2plY3RzL18vYnVja2V0cy9mYy1hb3UtZGF0YXNldHMtY29udHJvbGxlZA%3D%3D .


In [33]:
!gsutil -m -u $GOOGLE_PROJECT cp $admixture_estimates_q .

Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/admixture_estimates/aou_admixture_estimates_rye_v8.Q...
/ [1/1 files][ 32.4 MiB/ 32.4 MiB] 100% Done                                    
Operation completed over 1 objects/32.4 MiB.                                     


In [4]:
import os
import subprocess

destination_filename = 'aou_admixture_estimates_rye_v8.Q'

my_bucket = os.getenv('WORKSPACE_BUCKET')

args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

output.stderr

b'Copying file://./aou_admixture_estimates_rye_v8.Q [Content-Type=application/octet-stream]...\n/ [0 files][    0.0 B/ 32.4 MiB]                                                \r-\r- [0 files][  1.8 MiB/ 32.4 MiB]                                                \r\\\r\\ [0 files][  3.6 MiB/ 32.4 MiB]                                                \r|\r/\r/ [0 files][  5.4 MiB/ 32.4 MiB]                                                \r-\r- [0 files][  7.2 MiB/ 32.4 MiB]                                                \r\\\r|\r| [0 files][  9.0 MiB/ 32.4 MiB]                                                \r/\r/ [0 files][ 10.8 MiB/ 32.4 MiB]                                                \r-\r- [0 files][ 12.6 MiB/ 32.4 MiB]                                                \r\\\r|\r| [0 files][ 14.4 MiB/ 32.4 MiB]                                                \r/\r/ [0 files][ 16.2 MiB/ 32.4 MiB]                                                \r-\r\\\r\\ [0 files][ 18.0 MiB/ 32.4 MiB]    

In [6]:
import pandas as pd

name_of_file_in_bucket = 'aou_admixture_estimates_rye_v8.Q'

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
df_rye = pd.read_csv(name_of_file_in_bucket, sep='\t')
df_rye.head()

Copying gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/aou_admixture_estimates_rye_v8.Q...
/ [1 files][ 32.4 MiB/ 32.4 MiB]                                                
Operation completed over 1 objects/32.4 MiB.                                     


[INFO] aou_admixture_estimates_rye_v8.Q is successfully downloaded into your working space


,eur,eas,amr,afr,sas,mid,research_id
0,0.000000,0.002611,0.000000,0.940905,0.000000,0.056484,1000000
1,0.892052,0.000000,0.063450,0.000000,0.000000,0.044498,1000004
2,0.886335,0.000000,0.039803,0.000000,0.073862,0.000000,1000033
3,0.045004,0.003594,0.000000,0.874350,0.003528,0.073524,1000039
4,0.115799,0.014158,0.000000,0.853614,0.000000,0.016430,1000042


In [31]:
df_rye.head(30)

,eur,eas,amr,afr,sas,mid,research_id
0,0.000000,0.002611,0.000000,0.940905,0.000000,0.056484,1000000
1,0.892052,0.000000,0.063450,0.000000,0.000000,0.044498,1000004
2,0.886335,0.000000,0.039803,0.000000,0.073862,0.000000,1000033
3,0.045004,0.003594,0.000000,0.874350,0.003528,0.073524,1000039
4,0.115799,0.014158,0.000000,0.853614,0.000000,0.016430,1000042
5,0.000000,0.939365,0.000000,0.007806,0.031606,0.021223,1000045
6,0.781992,0.000000,0.120046,0.000000,0.028276,0.069686,1000059
7,0.906560,0.000000,0.019062,0.000000,0.046401,0.027977,1000061
8,0.783972,0.000952,0.091612,0.000000,0.042836,0.080627,1000062
9,0.894111,0.000000,0.050451,0.000000,0.041453,0.013986,1000070


# Relatedness

In [3]:
relatedness = f'{auxiliary_path}/relatedness'
relatedness

'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness'

In [4]:
related_samples_path = f'{relatedness}/relatedness_flagged_samples.tsv'

In [9]:
!gsutil -u $$GOOGLE_PROJECT cp $related_samples_path .

Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness_flagged_samples.tsv...
/ [1 files][239.0 KiB/239.0 KiB]                                                
Operation completed over 1 objects/239.0 KiB.                                    


In [2]:
destination_filename = 'relatedness_flagged_samples.tsv'

my_bucket = os.getenv('WORKSPACE_BUCKET')

args = ["gsutil", "cp", f"./{destination_filename}", f"{my_bucket}/Data/"]
output = subprocess.run(args, capture_output=True)

output.stderr

b'CommandException: No URLs matched: ./relatedness_flagged_samples.tsv\n'

In [3]:
name_of_file_in_bucket = 'relatedness_flagged_samples.tsv'

# get the bucket name
my_bucket = os.getenv('WORKSPACE_BUCKET')

# copy csv file from the bucket to the current working space
os.system(f"gsutil cp '{my_bucket}/Data/{name_of_file_in_bucket}' .")

print(f'[INFO] {name_of_file_in_bucket} is successfully downloaded into your working space')
# save dataframe in a csv file in the same workspace as the notebook
relatedness = pd.read_csv(name_of_file_in_bucket, sep='\t')
relatedness.head(30)

Copying gs://fc-secure-4f907dc3-1aa1-4aaa-8566-86d601589221/Data/relatedness_flagged_samples.tsv...
/ [1 files][239.0 KiB/239.0 KiB]                                                
Operation completed over 1 objects/239.0 KiB.                                    


[INFO] relatedness_flagged_samples.tsv is successfully downloaded into your working space


,sample_id
0,1000039
1,1000091
2,1000109
3,1000127
4,1000320
5,1000452
6,1000530
7,1000612
8,1000700
9,1000774


In [4]:
relatedness

,sample_id
0,1000039
1,1000091
2,1000109
3,1000127
4,1000320
...,...
30579,9996831
30580,9996887
30581,9997301
30582,9998505
